In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [52]:
import pandas as pd
moviesT = pd.read_csv('movies_cleanedFull.csv')

In [55]:
moviesT["tagged_description"].to_csv("tagged_description.txt", sep="\n", index=False, header=False)

In [56]:
raw_docs = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=447, chunk_overlap=0, separator="\n") #447 is max chunk size for tagged description
docs = text_splitter.split_documents(raw_docs)

In [58]:
db_movies = Chroma.from_documents(docs, embedding=OpenAIEmbeddings()) #create a vector data base

In [59]:
#test
query = "Movie about finances"
query_results = db_movies.similarity_search(query, k=10)
query_results

[Document(id='638f8e11-e7e3-4d93-be52-28ac7d99ff25', metadata={'source': 'tagged_description.txt'}, page_content='5397 A money troubled man finds some cash in an apartment which later turns out to belong to someone who stops at nothing to get his money back.\n5398 It follows four teenage girls living in a futuristic version of Lusaka, Zambia, who are recruited by a retired secret agent to save the world.'),
 Document(id='1467c57f-4f38-4afd-863d-a08e40c81fbc', metadata={'source': 'tagged_description.txt'}, page_content="6957 Three Wall Street bankers, a pair of crooks, and a millionaire are betrayed, manipulated and exploited by the efforts of a ruthless woman.\n6959 A feature documentary which captures Katharine Hepburn's spirit and determination, exploring her story using her own words, through a combination of hours of previously hidden and intimate audio tapes, video and photographic archive."),
 Document(id='a9904e94-4fcc-483e-acfb-31f7f1aee998', metadata={'source': 'tagged_descrip

In [60]:
moviesT[moviesT["movie_id"] == int(query_results[0].page_content.split()[0].strip())]

,movie_id,title,year,certificate,duration,genre,rating,description,stars,votes,words_in_description,tagged_description
5344,5397,Cash Only,(2015),Not Rated,91 min,"Crime, Drama, Thriller",6.0,A money troubled man finds some cash in an apa...,"['Malik Bader', '| ', ' Stars:', 'Nickola S...","1,246",27,5397 A money troubled man finds some cash in a...


In [61]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10
) -> pd.DataFrame:
    recs = db_movies.similarity_search(query, k=50)
    
    movies_list = []
    
    for i in range(0, len(recs)):
        movies_list += [int(recs[i].page_content.strip('"').split()[0])]
        
    return moviesT[moviesT["movie_id"].isin(movies_list)].head(top_k)

In [65]:
retrieve_semantic_recommendations(query)

,movie_id,title,year,certificate,duration,genre,rating,description,stars,votes,words_in_description,tagged_description
81,82,American Psycho,(2000),R,102 min,"Crime, Drama, Horror",7.6,A wealthy New York City investment banking exe...,"['Mary Harron', '| ', ' Stars:', 'Christian...","598,117",29,82 A wealthy New York City investment banking ...
266,267,The House,(III) (2022),TV-MA,97 min,"Animation, Comedy, Drama",6.8,"Across different eras, a poor family, an anxio...","['Paloma Baeza, ', 'Emma De Swaef, ', 'Niki Li...","15,290",25,"267 Across different eras, a poor family, an a..."
887,889,Heist,(III) (2015),R,93 min,"Action, Crime, Drama",6.1,A father is without the means to pay for his d...,"['Scott Mann', '| ', ' Stars:', 'Robert De ...","30,618",38,889 A father is without the means to pay for h...
918,920,Skandal! Bringing Down Wirecard,(2022),TV-14,92 min,"Documentary, Crime",7.2,Upstart payment firm Wirecard wowed the financ...,['James Erskine'],595,22,920 Upstart payment firm Wirecard wowed the fi...
1139,1143,On My Block,(2018–2021),TV-14,30 min,"Comedy, Drama",7.9,"A coming-of-age story about four bright, stree...","['Sierra Capri, ', 'Jason Genao, ', 'Brett Gra...","16,542",37,"1143 A coming-of-age story about four bright, ..."
1190,1194,The Laundromat,(I) (2019),R,96 min,"Comedy, Crime, Drama",6.3,"A widow investigates an insurance fraud, chasi...","['Steven Soderbergh', '| ', ' Stars:', 'Gar...","50,985",21,"1194 A widow investigates an insurance fraud, ..."
1638,1645,Scrooge: A Christmas Carol,(2022),NaN,96 min,"Animation, Adventure, Comedy",7.6,"A supernatural, time-travelling, musical adapt...","['Stephen Donnelly', '| ', ' Stars:', 'Jess...",NaN,11,"1645 A supernatural, time-travelling, musical ..."
1639,1646,Marauders,(2016),R,107 min,"Action, Crime, Thriller",5.5,"When a bank is hit by a brutal heist, all evid...","['Steven C. Miller', '| ', ' Stars:', 'Bruc...","22,443",48,"1646 When a bank is hit by a brutal heist, all..."
1694,1701,The Quintessential Quintuplets,(2019– ),TV-PG,24 min,"Animation, Comedy, Drama",7.7,"A poor, straight A student is hired to tutor s...","['Yoshitsugu Matsuoka, ', 'Kana Hanazawa, ', '...","4,864",34,"1701 A poor, straight A student is hired to tu..."
1893,1900,In the Land of Saints and Sinners,2017,NaN,136,Thriller,7.6,"In a remote Irish village, a damaged Finbar is...","['Robert Lorenz', '| ', ' Stars:', 'Liam Ne...",NaN,39,"1900 In a remote Irish village, a damaged Finb..."
